---
## 1. 基金分类（智能体1）

In [11]:
import sys
import os
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
import matplotlib
# 确保使用 inline 后端，避免 plt.show() 阻塞
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from math import pi
import matplotlib.font_manager as fm
target_fonts = ['Arial Unicode MS', 'PingFang SC', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC']
available_fonts = {f.name for f in fm.fontManager.ttflist}

matched_font = None
for font in target_fonts:
    if font in available_fonts:
        matched_font = font
        break

if matched_font:
    plt.rcParams['font.sans-serif'] = [matched_font, 'DejaVu Sans']
else:
    # 如果都没找到，尝试强制调用系统自带的通用黑体
    plt.rcParams['font.sans-serif'] = ['sans-serif', 'DejaVu Sans']

plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
plt.rcParams['font.size'] = 10              # 设置一个美观的基础字号
# ------------------------------

# 直接导入 .py 模块
module_dir = os.path.join(os.getcwd(), 'models/agents/fund_recommendation')
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

from fund_data import generate_funds
from agent1_classifier import FundClassifier
from agent2_pair_selector import PairSelector
from agent3_preference_learner import PreferenceLearner

print(f"当前使用的绘图字体: {matched_font if matched_font else '系统默认'}")

当前使用的绘图字体: Arial Unicode MS


In [12]:
print('正在生成并分析基金数据...')
funds_df = generate_funds(funds_per_category=10, seed=42)
classifier = FundClassifier()
classifier.fit(funds_df)
funds_df = classifier.funds_df

print(f'共 {len(funds_df)} 只基金，分为 4 类\n')
display(classifier.get_category_summary())

正在生成并分析基金数据...
共 40 只基金，分为 4 类



,类别,特征,基金数量,平均年化收益,平均年化标准差,平均夏普比率
0,优质型,高收益低风险,6,0.2539,0.1834,1.4222
1,增长型,高收益高风险,4,0.2723,0.2618,1.0639
2,稳健型,低收益低风险,20,0.1127,0.0654,2.0335
3,风险型,低收益高风险,10,0.0603,0.2234,0.2949


---
## 2. 交互式偏好学习

运行下方单元格启动交互——系统成对展示基金，您选择更偏好的那只。

In [13]:
def draw_radar_chart(fund_a, fund_b, funds_df):
    """绘制两只基金的三角形雷达图对比。"""
    dimensions = ['风控能力', '收益排名', '夏普排名']
    N = len(dimensions)

    all_returns = funds_df['annual_return'].values
    all_stds = funds_df['annual_std'].values
    all_sharpes = funds_df['sharpe_ratio'].values

    def global_rank(val, arr, reverse=False):
        """全市场百分位排名，映射到 0-10。reverse=True 表示值越小排名越高。"""
        if reverse:
            rank = (arr > val).mean()
        else:
            rank = (arr < val).mean()
        return rank * 10

    scores_a = [
        global_rank(fund_a['annual_std'], all_stds, reverse=True),
        global_rank(fund_a['annual_return'], all_returns),
        global_rank(fund_a['sharpe_ratio'], all_sharpes),
    ]
    scores_b = [
        global_rank(fund_b['annual_std'], all_stds, reverse=True),
        global_rank(fund_b['annual_return'], all_returns),
        global_rank(fund_b['sharpe_ratio'], all_sharpes),
    ]

    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    scores_a += scores_a[:1]
    scores_b += scores_b[:1]

    # 动态 y 轴范围，从两基金最低分往下留 1.5 分余量，避免图形塌缩成直线
    all_scores = scores_a[:-1] + scores_b[:-1]
    y_min = max(0, min(all_scores) - 1.5)
    y_max = 10

    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dimensions, fontsize=10)
    ax.set_ylim(y_min, y_max)
    y_ticks = np.linspace(y_min, y_max, 5)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f'{t:.1f}' for t in y_ticks], fontsize=7, color='grey')

    ax.plot(angles, scores_a, 'o-', linewidth=2, color='#2196F3',
            label=f'A: {fund_a["name"]}')
    ax.fill(angles, scores_a, alpha=0.1, color='#2196F3')
    ax.plot(angles, scores_b, 'o-', linewidth=2, color='#FF5722',
            label=f'B: {fund_b["name"]}')
    ax.fill(angles, scores_b, alpha=0.1, color='#FF5722')

    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)
    ax.set_title('基金对比雷达图', pad=20, fontsize=11, fontweight='bold')
    plt.tight_layout()
    # 使用 display 而非 plt.show()，避免交互式后端阻塞
    display(fig)
    plt.close(fig)


def interactive_learning():
    """交互式偏好学习。"""
    # 颜色常量，与雷达图一致 (A=蓝, B=橙)
    C_A = '\033[38;2;33;150;243m'   # #2196F3
    C_B = '\033[38;2;255;87;34m'    # #FF5722
    C_R = '\033[0m'

    print('=' * 56)
    print('  交互式偏好学习')
    print('=' * 56)
    print()

    learner = PreferenceLearner()
    # 候选对12个，每轮加速
    pair_selector = PairSelector(n_candidates=12)

    print('每轮展示两只基金，请选择您更偏好的那只 (A/B)。\n')
    input('按 Enter 键开始...')

    while not learner.is_converged():
        iteration = learner.n_iterations + 1

        # 第一步：计算（此时用户还看着上一轮的内容）
        (idx_i, idx_j), eig = pair_selector.select_pair(funds_df, learner)
        fund_i = funds_df.loc[idx_i]
        fund_j = funds_df.loc[idx_j]

        # 第二步：清屏并展示本轮全部内容（wait=False 立即清，不等待）
        clear_output(wait=False)

        # 先显示之前的学习进度（如果有）
        if learner.n_iterations > 0:
            params = learner.get_parameter_estimates()
            print(f'>> 已学习 {learner.n_iterations} 轮')
            print(f'   收益偏好 alpha = {params["alpha_mean"]:.2f} '
                  f'+/- {params["alpha_std"]:.2f}')
            print(f'   风险厌恶 beta  = {params["beta_mean"]:.2f} '
                  f'+/- {params["beta_std"]:.2f}')
            # 每3轮显示类别偏好
            if iteration % 3 == 0:
                cat_ranking = learner.get_category_expected_utility(funds_df)
                cats_str = ' | '.join(
                    f'{cr["category"]}: {cr["avg_expected_utility"]:.3f}'
                    for cr in cat_ranking
                )
                print(f'   类别偏好: {cats_str}')
            print()

        # 当前轮基金对（print 中的颜色与雷达图一致）
        print(f'--- 第 {iteration} 轮比较 ---\n')
        print(f'  {C_A}A{C_R}. [{fund_i["category"]}] {C_A}{fund_i["name"]}{C_R}')
        print(f'     年化收益: {fund_i["annual_return"]:.2%}  '
              f'|  年化标准差: {fund_i["annual_std"]:.2%}')
        print(f'     夏普比率: {fund_i["sharpe_ratio"]:.2f}  '
              f'|  类型: {fund_i["category_label"]}')
        print()
        print(f'  {C_B}B{C_R}. [{fund_j["category"]}] {C_B}{fund_j["name"]}{C_R}')
        print(f'     年化收益: {fund_j["annual_return"]:.2%}  '
              f'|  年化标准差: {fund_j["annual_std"]:.2%}')
        print(f'     夏普比率: {fund_j["sharpe_ratio"]:.2f}  '
              f'|  类型: {fund_j["category_label"]}')
        print()

        # 绘制雷达图
        draw_radar_chart(fund_i, fund_j, funds_df)

        # 确保全部内容显示后才弹出输入框
        sys.stdout.flush()

        # 获取用户选择（input 框不支持 ANSI，不加入颜色代码）
        choice = None
        while choice not in [0, 1]:
            inp = input('请输入您的选择 (A/B)，或 Q 退出: ').strip().upper()
            if inp == 'Q':
                print('\n感谢使用！')
                return
            elif inp == 'A':
                choice = 0
            elif inp == 'B':
                choice = 1
            else:
                print('请输入 A 或 B')

        # 更新后验
        learner.update(funds_df.loc[idx_i], funds_df.loc[idx_j], choice)

    # ====== 推荐结果 ======
    clear_output(wait=False)
    print('=' * 56)
    print('  推荐结果')
    print('=' * 56)
    print(f'\n经过 {learner.n_iterations} 轮交互，系统已充分了解您的偏好！\n')

    params = learner.get_parameter_estimates()
    print('您的投资画像:')
    print(f'  - 收益偏好 alpha = {params["alpha_mean"]:.2f}')
    print(f'  - 风险厌恶 beta  = {params["beta_mean"]:.2f}')
    if params['beta_mean'] > params['alpha_mean'] * 1.2:
        print('  >> 您属于 风险规避型，更注重资金安全性')
    elif params['alpha_mean'] > params['beta_mean'] * 1.2:
        print('  >> 您属于 收益追求型，更看重投资回报')
    else:
        print('  >> 您属于 均衡型，在收益和风险之间寻求平衡')

    print('\n各类基金匹配度排名:')
    cat_ranking = learner.get_category_expected_utility(funds_df)
    for rank, cr in enumerate(cat_ranking, 1):
        bar = '#' * max(1, int(cr['avg_expected_utility'] * 20 + 5))
        print(f'  {rank}. {cr["category"]} ({cr["category_label"]}):  '
              f'{bar}  {cr["avg_expected_utility"]:.4f}')

    rec = learner.recommend(funds_df, top_n=3)
    print(f'\n** 最佳匹配: {rec["best_category"]["category"]}'
          f'（{rec["best_category"]["category_label"]}）')
    print(f'\n为您推荐的前 3 只基金:\n')
    for rank, fund in enumerate(rec['top_funds'], 1):
        print(f'  {rank}. [{fund["category"]}] {fund["name"]}')
        print(f'     年化收益: {fund["annual_return"]:.2%}  '
              f'|  年化标准差: {fund["annual_std"]:.2%}')
        print()

    # 最终推荐的雷达图
    if len(rec['top_funds']) >= 2:
        print('\n前两只推荐基金的对比雷达图:')
        draw_radar_chart(rec['top_funds'][0], rec['top_funds'][1], funds_df)

    sys.stdout.flush()


interactive_learning()

  交互式偏好学习

每轮展示两只基金，请选择您更偏好的那只 (A/B)。



KeyboardInterrupt: Interrupted by user